In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn as nn
import sklearn.preprocessing
import os
import time
import glob
import json
import xgboost as xgb
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

**For the first training run, we will train models on "G_low_vol_mid_liq" in our strata**

In [2]:
with open('stratified_metadata.json', 'r') as f:
    metadata = json.load(f)

STRATUM = 'G_low_vol_mid_liq'
paths = metadata[STRATUM]

FEATURE_COLS = ['SMA_20', 'EMA_10', 'RSI_14', 'ROC_10', 'VOL_20', 'VOLUME_20_D', 'SKEW_20', 'KURT_20']

dfs = []
for path in paths:
    try:
        df = pd.read_parquet(path)
        df['ticker'] = os.path.splitext(os.path.basename(path))[0]
        dfs.append(df)
    except Exception as e:
        print(f"Skipping {path}: {e}")

panel = pd.concat(dfs)
panel.index.name = 'date'
panel = panel.reset_index().sort_values('date').reset_index(drop=True)

print(f"Loaded {panel['ticker'].nunique()} tickers, {len(panel)} rows")
print(f"Date range: {panel['date'].min().date()} → {panel['date'].max().date()}")

Loaded 265 tickers, 99682 rows
Date range: 2024-04-01 → 2026-03-02


**Cross-sectional normalization**

For each trading day, z-score each feature *across all tickers in the stratum* — this captures relative strength vs peers and is leak-free since each day's normalization only uses that day's cross-section. The target is converted to a within-day percentile rank for the same reason.

In [3]:
for col in FEATURE_COLS:
    panel[col] = panel.groupby('date')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

# Rank target within each day's cross-section → percentile [0, 1]
# Model learns "which stocks outperform peers" rather than absolute return direction
panel['Target'] = panel.groupby('date')['Target'].rank(pct=True)

# Drop any rows where all features are NaN (e.g. days with only one ticker present)
panel = panel.dropna(subset=FEATURE_COLS + ['Target']).reset_index(drop=True)

print("Cross-sectional normalization complete.")
panel[FEATURE_COLS + ['Target']].describe().round(3)

Cross-sectional normalization complete.


,SMA_20,EMA_10,RSI_14,ROC_10,VOL_20,VOLUME_20_D,SKEW_20,KURT_20,Target
count,99682.000,99682.000,99682.000,99682.000,99682.000,99682.000,99682.000,99682.000,99682.000
mean,0.000,0.000,0.000,-0.000,-0.000,-0.000,-0.000,0.000,0.502
std,0.998,0.998,0.998,0.998,0.998,0.998,0.998,0.998,0.289
min,-15.241,-15.602,-5.453,-14.682,-1.677,-1.579,-7.997,-2.761,0.004
25%,-0.437,-0.432,-0.645,-0.439,-0.557,-0.443,-0.631,-0.566,0.253
50%,-0.049,-0.034,-0.036,-0.048,-0.129,-0.209,-0.048,-0.214,0.503
75%,0.397,0.398,0.628,0.391,0.317,0.109,0.573,0.289,0.753
max,16.053,15.990,5.764,16.097,16.157,13.059,8.260,12.639,1.000


**Temporal train/test split**

Cut on the 80th-percentile date so all training samples precede all test samples — no shuffling.

In [4]:
dates = np.sort(panel['date'].unique())
cutoff_date = dates[int(len(dates) * 0.8)]

train = panel[panel['date'] < cutoff_date]
test  = panel[panel['date'] >= cutoff_date]

X_train = train[FEATURE_COLS].values
y_train = train['Target'].values
X_test  = test[FEATURE_COLS].values
y_test  = test['Target'].values

print(f"Train: {train['date'].min().date()} → {train['date'].max().date()}  ({len(train)} rows, {train['ticker'].nunique()} tickers)")
print(f"Test:  {test['date'].min().date()} → {test['date'].max().date()}  ({len(test)} rows, {test['ticker'].nunique()} tickers)")

Train: 2024-04-01 → 2025-10-09  (74160 rows, 256 tickers)
Test:  2025-10-10 → 2026-03-02  (25522 rows, 265 tickers)


**Model training and evaluation**

Two models trained on the same prepared data: Lasso (linear baseline) and XGBoost (non-linear). Both are evaluated with MSE and Information Coefficient (IC) — the Spearman rank correlation between predicted and actual percentile ranks, which is the standard metric for cross-sectional return prediction.

In [5]:
from scipy.stats import spearmanr

def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    ic, p = spearmanr(y_true, y_pred)
    print(f"{name:10s}  MSE: {mse:.4f}  IC: {ic:.4f}  (p={p:.3f})")

# --- Lasso ---
lasso = Lasso(alpha=0.01, max_iter=5000)
lasso.fit(X_train, y_train)
evaluate("Lasso", y_test, lasso.predict(X_test))

# --- XGBoost ---
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
evaluate("XGBoost", y_test, xgb_model.predict(X_test))

# --- Feature importance (XGBoost) ---
importances = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print(f"\nXGBoost feature importances:\n{importances.round(4).to_string()}")

C:\Users\dhruv\AppData\Local\Temp\ipykernel_21092\1289151723.py:5: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, p = spearmanr(y_true, y_pred)


Lasso       MSE: 0.0833  IC: nan  (p=nan)
XGBoost     MSE: 0.0832  IC: 0.0428  (p=0.000)

XGBoost feature importances:
VOL_20         0.1400
EMA_10         0.1354
ROC_10         0.1274
SMA_20         0.1265
KURT_20        0.1227
SKEW_20        0.1222
RSI_14         0.1204
VOLUME_20_D    0.1054


**DNN and LSTM**

The DNN operates on the same flat (ticker, date) rows as Lasso/XGBoost. The LSTM needs sequential input — for each ticker we build rolling windows of `SEQ_LEN` days so the model can learn temporal patterns. Both share the same evaluate() helper and are compared on the same test period.

In [6]:
import torch
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

SEQ_LEN    = 20
EPOCHS     = 50
BATCH_SIZE = 256
LR         = 1e-3

# ── Flat tensors (DNN uses same X_train/X_test as sklearn models) ──────────
X_tr_flat = torch.FloatTensor(X_train)
y_tr_flat = torch.FloatTensor(y_train)
X_te_flat = torch.FloatTensor(X_test)
y_te_flat = torch.FloatTensor(y_test)

# ── Sequence builder for LSTM ──────────────────────────────────────────────
def make_sequences(df, seq_len):
    X_seqs, y_seqs, seq_dates = [], [], []
    for ticker, group in df.groupby('ticker'):
        group = group.sort_values('date')
        feats   = group[FEATURE_COLS].values.astype(np.float32)
        targets = group['Target'].values.astype(np.float32)
        dates   = group['date'].values
        for i in range(seq_len, len(group)):
            X_seqs.append(feats[i - seq_len:i])
            y_seqs.append(targets[i])
            seq_dates.append(dates[i])
    return np.array(X_seqs), np.array(y_seqs), np.array(seq_dates)

X_seq, y_seq, seq_dates = make_sequences(panel, SEQ_LEN)

train_mask = seq_dates < cutoff_date
test_mask  = seq_dates >= cutoff_date

X_tr_seq = torch.FloatTensor(X_seq[train_mask])
y_tr_seq = torch.FloatTensor(y_seq[train_mask])
X_te_seq = torch.FloatTensor(X_seq[test_mask])
y_te_seq = torch.FloatTensor(y_seq[test_mask])

dnn_loader  = DataLoader(TensorDataset(X_tr_flat, y_tr_flat), batch_size=BATCH_SIZE, shuffle=True)
lstm_loader = DataLoader(TensorDataset(X_tr_seq,  y_tr_seq),  batch_size=BATCH_SIZE, shuffle=True)

print(f"DNN  — train: {X_tr_flat.shape}, test: {X_te_flat.shape}")
print(f"LSTM — train: {X_tr_seq.shape},  test: {X_te_seq.shape}")

Using device: cuda
DNN  — train: torch.Size([74160, 8]), test: torch.Size([25522, 8])
LSTM — train: torch.Size([69125, 20, 8]),  test: torch.Size([25257, 20, 8])


In [7]:
# ── Model definitions ──────────────────────────────────────────────────────
class DNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),          nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)  # last timestep → prediction

# ── Generic training loop ──────────────────────────────────────────────────
def train_torch(model, loader, epochs, lr):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"  epoch {epoch+1:3d}/{epochs}  loss: {epoch_loss/len(loader):.4f}")
    return model

def eval_torch(name, model, X_te, y_te):
    model.eval()
    with torch.no_grad():
        preds = model(X_te.to(device)).cpu().numpy()
    evaluate(name, y_te.numpy(), preds)

# ── Train and evaluate ─────────────────────────────────────────────────────
print("── DNN ──────────────────────────────")
dnn = train_torch(DNN(len(FEATURE_COLS)), dnn_loader, EPOCHS, LR)
eval_torch("DNN", dnn, X_te_flat, y_te_flat)

print("\n── LSTM ─────────────────────────────")
lstm = train_torch(LSTMModel(len(FEATURE_COLS)), lstm_loader, EPOCHS, LR)
eval_torch("LSTM", lstm, X_te_seq, y_te_seq)

── DNN ──────────────────────────────
  epoch  10/50  loss: 0.0837
  epoch  20/50  loss: 0.0829
  epoch  30/50  loss: 0.0828
  epoch  40/50  loss: 0.0825
  epoch  50/50  loss: 0.0823
DNN         MSE: 0.0835  IC: 0.0450  (p=0.000)

── LSTM ─────────────────────────────
  epoch  10/50  loss: 0.0825
  epoch  20/50  loss: 0.0805
  epoch  30/50  loss: 0.0760
  epoch  40/50  loss: 0.0712
  epoch  50/50  loss: 0.0680
LSTM        MSE: 0.0885  IC: 0.0108  (p=0.086)
